# Bagian 1: Mendapatkan API Key


In [1]:
!pip install requests python-dotenv --quiet


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import requests
import pandas as pd
import time
import os
from dotenv import load_dotenv

load_dotenv()  # membaca isi file .env

API_KEY = os.getenv("COIN_API_KEY")
    
if API_KEY:
    print("API key berhasil dimuat.")
else:
    print("API key belum ketemu. Pastikan file .env sudah dibuat dan diisi dengan benar.")

API key berhasil dimuat.


# Bagian 2: Mencoba Memanggil API

In [3]:
alamat_api = "https://api.coingecko.com/api/v3/coins/markets"

parameter = {
    "vs_currency": "usd",
    "per_page": 5,
    "page": 1
}
headers = {
    "accept": "application/json",
    "x-cg-demo-api-key": API_KEY
}

response = requests.get(alamat_api, params=parameter, headers=headers)
print(f"Status Code: {response.status_code}")

hasil = response.json()
print(f"Jumlah koin yang dikirim kali ini: {len(hasil)}")

# Melihat bentuk data koin pertama
koin_pertama = hasil[0]
print("\nNama Koin :", koin_pertama["name"])
print("Harga USD :", koin_pertama["current_price"])
print("Update    :", koin_pertama["last_updated"])

Status Code: 200
Jumlah koin yang dikirim kali ini: 5

Nama Koin : Bitcoin
Harga USD : 85549
Update    : 2026-09-23T11:46:00.000Z


In [4]:
# Ambil data koin pertama (indeks ke-0 dari list)
koin_pertama = hasil[0]

# Tampilkan datanya dengan key milik CoinGecko
print("Nama Koin :", koin_pertama["name"])
print("Harga USD :", koin_pertama["current_price"])
print("Tanggal   :", koin_pertama["last_updated"])

Nama Koin : Bitcoin
Harga USD : 85549
Tanggal   : 2026-09-23T11:46:00.000Z


# Bagian 3: Membungkus Jadi Class


In [5]:
class KlienKripto:

    def __init__(self, api_key):
        self.api_key = api_key
        self.alamat_api = "https://api.coingecko.com/api/v3/coins/markets"

    def ambil_data(self, halaman, jumlah=50):
        parameter = {
            "vs_currency": "usd",
            "per_page": jumlah,
            "page": halaman
        }
        headers = {
            "accept": "application/json",
            "x-cg-demo-api-key": self.api_key
        }

        try:
            response = requests.get(self.alamat_api, params=parameter, headers=headers, timeout=20)
        except Exception:
            print("Koneksi bermasalah, mencoba lagi...")
            time.sleep(3)
            response = requests.get(self.alamat_api, params=parameter, headers=headers, timeout=20)

        if response.status_code != 200:
            print(f"Gagal mengambil data. Status: {response.status_code}")
            return pd.DataFrame()

        daftar_koin = response.json()

        data = []
        for koin in daftar_koin:
            data.append({
                "ID_Koin"            : koin["id"],
                "Nama"               : koin["name"],
                "Simbol"             : koin["symbol"],
                "Harga_USD"          : koin["current_price"],
                "Kapitalisasi_Pasar" : koin["market_cap"],
                "Terakhir_Diperbarui": koin["last_updated"]
            })

        return pd.DataFrame(data)

print("Class KlienKripto siap dipakai!")

Class KlienKripto siap dipakai!


# Mengambil 150 Koin dengan 3 Halaman



In [6]:
klien = KlienKripto(API_KEY)

daftar_halaman = [1, 2, 3] # Mengambil 3 halaman
semua_tabel = []

for halaman in daftar_halaman:
    # 50 koin per halaman x 3 halaman = 150 koin
    tabel = klien.ambil_data(halaman=halaman, jumlah=50)
    print(f"Halaman {halaman}: {len(tabel)} koin diambil")
    semua_tabel.append(tabel)
    time.sleep(1) # Jeda agar tidak terkena limit API

df_kripto = pd.concat(semua_tabel, ignore_index=True)

print(f"\nTotal koin terkumpul: {len(df_kripto)}")
df_kripto.head()

Halaman 1: 50 koin diambil
Halaman 2: 50 koin diambil
Halaman 3: 50 koin diambil

Total koin terkumpul: 150


,ID_Koin,Nama,Simbol,Harga_USD,Kapitalisasi_Pasar,Terakhir_Diperbarui
0,bitcoin,Bitcoin,btc,85549.000000,1717865348538,2026-09-23T11:46:00.000Z
1,ethereum,Ethereum,eth,2721.240000,331950797169,2026-09-23T11:46:00.000Z
2,tether,Tether,usdt,0.999744,183419227699,2026-09-23T11:46:00.000Z
3,binancecoin,BNB,bnb,782.400000,104108886566,2026-09-23T11:46:00.000Z
4,ripple,XRP,xrp,1.570000,98834184494,2026-09-23T11:46:00.000Z


# Bagian 4: Membersihkan Data

### 1. Identifikasi Tipe Data Awal

Melihat tipe data dari DataFrame yang baru diambil dari API.

In [7]:
print("Tipe data awal:")
print(df_kripto.dtypes)

Tipe data awal:
ID_Koin                 object
Nama                    object
Simbol                  object
Harga_USD              float64
Kapitalisasi_Pasar       int64
Terakhir_Diperbarui     object
dtype: object


### 2. Identifikasi dan Penghapusan Data Duplikat

In [8]:
jumlah_sebelum = len(df_kripto)
jumlah_duplikat = df_kripto.duplicated(subset="ID_Koin").sum()
print(f"Jumlah baris duplikat ditemukan: {jumlah_duplikat}")

if jumlah_duplikat > 0:
    df_bersih = df_kripto.drop_duplicates(subset="ID_Koin").copy()
    print(f"Baris kembar dibuang : {jumlah_sebelum - len(df_bersih)}")
else:
    df_bersih = df_kripto.copy()
    print("Tidak ada baris kembar yang dibuang.")

Jumlah baris duplikat ditemukan: 0
Tidak ada baris kembar yang dibuang.


### 3. Penanganan Nilai Kosong

In [9]:
# Cek nilai kosong
print("Jumlah nilai kosong pada tiap kolom:")
print(df_bersih.isnull().sum())

# Isi Kapitalisasi_Pasar jika ada yang kosong dengan 0
def bersihkan_kapitalisasi(nilai):
    if pd.isna(nilai):
        return 0
    return nilai

df_bersih["Kapitalisasi_Pasar"] = df_bersih["Kapitalisasi_Pasar"].apply(bersihkan_kapitalisasi)

Jumlah nilai kosong pada tiap kolom:
ID_Koin                0
Nama                   0
Simbol                 0
Harga_USD              0
Kapitalisasi_Pasar     0
Terakhir_Diperbarui    0
dtype: int64


### 4. Perubahan Tipe Data

Mengubah kolom tanggal dan memastikan kolom angka bertipe numerik.

In [10]:
def ubah_ke_tanggal(teks):
    return pd.to_datetime(teks)

# Ubah tipe data waktu
df_bersih["Terakhir_Diperbarui"] = df_bersih["Terakhir_Diperbarui"].apply(ubah_ke_tanggal)

# Pastikan Harga dan Kapitalisasi_Pasar bertipe numerik
df_bersih["Harga_USD"] = pd.to_numeric(df_bersih["Harga_USD"])
df_bersih["Kapitalisasi_Pasar"] = pd.to_numeric(df_bersih["Kapitalisasi_Pasar"])

print("Tipe data setelah perubahan:")
print(df_bersih.dtypes)
df_bersih.head()

Tipe data setelah perubahan:
ID_Koin                             object
Nama                                object
Simbol                              object
Harga_USD                          float64
Kapitalisasi_Pasar                   int64
Terakhir_Diperbarui    datetime64[ns, UTC]
dtype: object


,ID_Koin,Nama,Simbol,Harga_USD,Kapitalisasi_Pasar,Terakhir_Diperbarui
0,bitcoin,Bitcoin,btc,85549.000000,1717865348538,2026-09-23 11:46:00+00:00
1,ethereum,Ethereum,eth,2721.240000,331950797169,2026-09-23 11:46:00+00:00
2,tether,Tether,usdt,0.999744,183419227699,2026-09-23 11:46:00+00:00
3,binancecoin,BNB,bnb,782.400000,104108886566,2026-09-23 11:46:00+00:00
4,ripple,XRP,xrp,1.570000,98834184494,2026-09-23 11:46:00+00:00


### Pemeriksaan Terakhir

In [11]:
print(f"Jumlah baris akhir     : {len(df_bersih)}")
print(f"Sudah >= 100 baris?    : {len(df_bersih) >= 100}")
print(f"ID_Koin kembar         : {df_bersih['ID_Koin'].duplicated().sum()}")

Jumlah baris akhir     : 150
Sudah >= 100 baris?    : True
ID_Koin kembar         : 0


---

# Bagian 5: Menyimpan Hasil

Menyimpan dataset yang sudah bersih menjadi file CSV.

In [12]:
nama_file = "dataset_kripto.csv"
df_bersih.to_csv(nama_file, index=False)
print(f"Data berhasil disimpan ke file: {nama_file}")

df_cek = pd.read_csv(nama_file)
print(f"File terbaca kembali: {len(df_cek)} baris, {len(df_cek.columns)} kolom")
df_cek.head()

Data berhasil disimpan ke file: dataset_kripto.csv
File terbaca kembali: 150 baris, 6 kolom


,ID_Koin,Nama,Simbol,Harga_USD,Kapitalisasi_Pasar,Terakhir_Diperbarui
0,bitcoin,Bitcoin,btc,85549.000000,1717865348538,2026-09-23 11:46:00+00:00
1,ethereum,Ethereum,eth,2721.240000,331950797169,2026-09-23 11:46:00+00:00
2,tether,Tether,usdt,0.999744,183419227699,2026-09-23 11:46:00+00:00
3,binancecoin,BNB,bnb,782.400000,104108886566,2026-09-23 11:46:00+00:00
4,ripple,XRP,xrp,1.570000,98834184494,2026-09-23 11:46:00+00:00


---

# Bagian 6: Bahan untuk Slide

Informasi yang bisa dicopy untuk dipindahkan ke slide presentasi.

In [13]:
print("=" * 50)
print("ANGKA UNTUK SLIDE")
print("=" * 50)
print("Sumber data      : CoinGecko API")
print("Data diambil     : Top cryptocurrency by market cap (USD)")
print()
print(f"Baris sebelum dibersihkan : {len(df_kripto)}")
print(f"Baris dataset akhir       : {len(df_bersih)}")
print()
print("Class yang dibuat:")
print("  1. KlienKripto - mengambil data pasar koin dari CoinGecko")
print()
print("Function yang dibuat:")
print("  1. bersihkan_kapitalisasi - mengisi kapitalisasi kosong dengan 0")
print("  2. ubah_ke_tanggal        - mengubah format string ISO ke datetime")
print()
print("Temuan dari pembersihan data:")
print(f"  Baris kembar dibuang     : {jumlah_sebelum - len(df_bersih)}")
print("=" * 50)

ANGKA UNTUK SLIDE
Sumber data      : CoinGecko API
Data diambil     : Top cryptocurrency by market cap (USD)

Baris sebelum dibersihkan : 150
Baris dataset akhir       : 150

Class yang dibuat:
  1. KlienKripto - mengambil data pasar koin dari CoinGecko

Function yang dibuat:
  1. bersihkan_kapitalisasi - mengisi kapitalisasi kosong dengan 0
  2. ubah_ke_tanggal        - mengubah format string ISO ke datetime

Temuan dari pembersihan data:
  Baris kembar dibuang     : 0


---

# Ringkasan

| Aspek | Ada di bagian |
|---|---|
| Pengambilan data lewat API | Bagian 2 dan 3 |
| Struktur OOP | Bagian 3, class `KlienKripto` |
| Function dan modularitas | Bagian 4, `bersihkan_kapitalisasi` dan `ubah_ke_tanggal` |
| Data cleaning | Bagian 4 |
| Minimal 100 baris | Dicek di akhir Bagian 3 dan Bagian 4 |
